# About

This notebook for documenting my process of understanding the paper *SAM-Body4D: Training-Free 4D Human Body Mesh Recovery from Videos* by Mingqui Gao et al. The aim is to provide a high-level overview of the algorithm.

# Motivation

Human Mesh Recovery (HMR) aims to reconstruct 3D human pose from 2D visual observations. Recent advances in image-based HMR, such as SAM 3D Body, have promised a robust and generalisable approach to reconstructing the human body from diverse and complex in-the-wild images. However, when extended to videos, most image-based HMR methods operate in a frame-by-frame manner, leading to a lack of temporal continuity and the reconstructed human meshse often fluctuate and fail to remain stable in video scenarios. This becomes particularly problematic when there are dynamic camera motion, back-ground clutter, and frequent occulsions. Existing video-based HMR meothods attempts to address this by modelling temporal information, however, they are fundamentally optimisation based, demanding large amounts of manually annotated video data and carefully tuned loss functions. Such reliance restricts their generalisablility and weakens the robustness to diverse and unpredictable human motions (interesting!). 

Building upon this insight, they proposed SAM-Body4D, which is a temporally consistent HMR algorithm from videos. Importantly, it is a training-free framework, which will be discussed later, but basically it enforces temporal coherence by directly leveraging the pixel-level continutiy of the human in the video (rather than relying on feature-space temporal modelling where such continuity may already be lost). 

# Preliminary

SAM 3 (Segment Anything Model 3) is Meta's unified foundation model for promptable segmentation. SAM 3D Body is part of the SAM 3D which is Meta's unified 3D reconstruction family, tailoered specifically for 3D human mesh reconstruction.

## SAM 3

SAM 3 is a promptable *object segmentation model* for both images and videos (It's an exntetion of SAM and SAM2. SAM demonstrates high segmentation accuracy for images by leveraging billion-scale training data and strong transformer architectures. SAM2 extends these strengths to the video sestting by introducing a memory mechanism). Hahah, this is the first time seeing videos being mathematically represented.

Let \\( \mathcal{V} = \{I_t\}_{t=1}^T, I_t \in \mathbb{R}^{H \times W \times 3} \\) denote a video (a sequence of images) and \\(\mathcal{P}\\) denote user-defined prompts.  SAM3 predicts a *mask sequence* (masklet) defined as

\\[
\mathcal{M} = \{M_t\}_{t=1}^T, \quad \text{where } M_t \in \{0, 1\}^{H \times W}
\\]

Each mask \\(M_t\\) for the \\(t\\)-th frame indicates whether the pixel belongs to the foreground \\((1)\\) or the background \\((0)\\). It basically tells us which pixel belongs to the target human body. This is obtained by combining two complementary modules `propagate` and `detect`.

\begin{align*}
\hat{M_t} & = \text{propagate}(M_{t-1})\\
O_t & = \text{detect} (I_t, \mathcal{P})\\
M_t & = \text{match\_and\_update} ( \hat{M_t} , O_t )
\end{align*}

The `propagate` module leverages the spatial-temporal correspondence between the previous and current frame (prediction), ensuring a that the change in masks across time is reliable and preserves continuity. The `detect` module, on the other hand, tries to identify the object in the current frame based on the prompt. By combining the outputs of these two modules, SAM3 achieves substantially higher accuracy than SAM 2 and other VOS methods on complex videos. 

## SAM 3D Body

SAM 3D Body is a promptable *HMR model* for in-the-wild images. It takes an image as input, performs feature encoding, then decodes it to predict full-body tokens (toekns are like a box of information). Importantly, it accepts prompts at both the encoding and decoding stages. Consider an input image \\(I \in \mathbb{R}^{H \times W \times 3}\\), SAM 3D Body performs feature encoding by

\\[
F = \text{ImgEncoder}(I, \mathcal{P}_{\text{enc}})
\\]

where \\(\mathcal{P}_{\text{enc}}\\) denotes optional encoder prompts (like segmentation masks!) that helps the model fous on the target human. The encoder probably lifts the input vector into a higher dimensional latent space (probably a neural network?), the encoded features \\(F\\) contains lots of complex information.

With the encoded features, the decoder returns full-body tokens by

\\[
O = \text{Decoder}(F, \mathcal{P}_{\text{dec}})
\\]

where \\(\mathcal{P}_{\text{enc}}\\) denotes optional decoder prompts such as keypoint (the shoulder is here), camera (the camera is from this angle and distance) or MHR (body parameters) tokens.  Just like in many other transformer based models, the first token is designed to be like a master summary. Therefore, the first output token \\(O_0\\) is then passed through some neural network (multilayer preceptron) to obtain the momentum human rig (MHR) parameters:

\\[
\theta = \{ P, S, C, S_k\} = \text{MLP} (O_0)
\\]

where \\(P\\), \\(S\\), \\(C\\), and \\(S_k\\) denote pose, shape, camera pose, and skelton parameters.

Apparently the body and hand are otpimised separaetd  due to different optimisation strageters during the inference and are later fused. Unless otherwise specified, \\(\theta\\) indicates the final full-body mesh paramters containing both body and hand info.

# How does it work?

## Overview

Given an input video \\( \mathcal{V} = \{I_t\}_{t=1}^T\\) and prompts \\( \mathcal{P} = \{ \mathcal{P}^{h_i} \}_{i=1}^T \\) indicating N target humans, SAM Body4D estimates *temporally consistent* human mesh parameters \\( \Theta = \{ \theta_t^{h_i} \}_{t=1, i=1}^{T, N} \\) for all selected persons.

It consists of three key components:

- A *Masklet Generator*, which produces indentity-consistent masklets (sequence of masks) as tracking cues over time.
- An *Occlusion-Aware Masklet Refiner*, which enhances these maskets by recovering missing regions when occlusion occurs.
- A *Masked Guided MHR module*, which ten predicts the per-frame mesh parameters.

Note that since each mesh is 'aligned' with its corresponding mask over time, as long as we have temporal continuity in masklets, then that continuity is naturally propagated to the reconstructed human meshes.

## Masklet Generator

For each target human \\(h_i\\) specified by prompts \\(\mathcal{P}\\), they applied **SAM 3** over the video \\(\mathcal{V}\\) to obtain spatio-temporally aligned masklets (sequence of masks) \\(\mathcal{M} = \{ M_t^{h_i} \} \\). Note that identity consistency is maintained across frames following the hypbrid progagtion-detection formulation explained above.

## Occlusion-Aware Masklet Refiner

In in-the-wild videos, humans frequently undergo severe occlusions, where even state-of-the-art segmentation methods like SAM 3 can only capture visible body regions. Such incomplete masklets provide insufficient visual evidence for human mesh estimation and may lead to hallucinated predictions with unrealistic pose and shape. To resolve this issue, they introduced an occulusion-aware refinement module to recover missing regions and offer complete visual referencse for the subsequent HMR stage. 

This is done by feeding the video \\(\mathcal{V}\\) and masklets \\(\mathcal{M}\\) into a mask completion model (predicts what the entire person should look like, even the hidden parts), Diffusion-VAS (Chen et al., 20225), producing completed masklets \\( \mathcal{\tilde{M}} = \{ \tilde{M}_t^{h_i} \} \\). This is the frst pass that only detects the mask (1 or 0), in other words, a best guess silhouette.

\\[
M_t^{h_i} \leftarrow \tilde{M}_t^{h_i}
\\]

Next, occlusion is detected for each human \\(h_i\\) at frame \\(t\\) if the completed mask is larger and doesn't overlap well with the original. This is formulated with the indicator function below

\\[
\text{occ}(t, h_i) = \mathbb{1} \left( \mid \tilde{M}_t^{h_i} \mid > \mid M_t^{h_i} \mid \text{ AND } \ \text{IoU}(\tilde{M}_t^{h_i}, M_t^{h_i}) < 0.7\right)
\\]

Finally, if occlusion is detected, that human at that frame goes through a second round of diffusion-VAS. This is the second pass that recovers the actual pixels, which is important in the subsequent HMR module because it expects visual features, not a 2D mask. Therefore this time, the corresponding frames are updated,

\\[
I_t^{h_i} \leftarrow \tilde{I}_t^{h_i}
\\]

Hence, yielding a refined video \\(\tilde{\mathcal{V}}\\) and refined masklets \\(\tilde{\mathcal{M}}\\), which provide more reliable superbvision for the subsequent HMR module.

## Training-Free Mask-Guided HMR

With refined video \\(\tilde{\mathcal{V}}\\) and refined masklets \\(\tilde{\mathcal{M}}\\), they perform SAM 3D Body. For each target huamn \\(h_i\\), the corresponding mask \\(\tilde{M}_t^{h_i}\\) is used as the encoder prompt \\(\mathcal{P}_{\text{enc}}\\), encouraging the model to focus on the correct identity. This produces per-frame mesh parameters \\( \Theta = \{ \theta_t^{h_i}\}_{t=1, i=1}^{T, N} \\). IMportantly, this entire pipeline opreates in a training-free manner without any task-specific finetuning.

To further enhance motion stability, they applied lightweight test-time temporal smoothing (whatever that means) to the Momentum HUman Rig pose and hand parameters, reducing jitter and promoting smooth transtions. 

Additionally, for each target human, the scale and shape parameters from the first (visible) frame is reused across the entire sequence to maintain consistent body proportions and avoid identity drift. These operations require no learning and introduce neglibile computation overhead, keeping the entire gramwork fully training-free. 

The full procedure of SAM-Body4D is explained in Algorithm 1, which explicitly details the execution flow. Check that if needed.

# Paper vs implementation

In the actual implementation there are two additional models, or "checkpoints", required.

The first is MoGe-2, which is used as the `fov_estimator` for SAM 3D Body. It provides a 3D point cloud of the scene from an image, developed by Microsoft. 

The second is DepthAnything V2, which is used as the `depth-encoder` for Diffusion-VAS. It provides a depth map from an image.

Furthermore, Detectron2 also needed to be installed (this one is a library). This was because ViTDet (Vision Transformer Detector, which is built on top of Detectron2) was used as the `huamn_detector`. This is because SAM 3 focuses on segmentation, so it needs a detector that tells who to track in the first place.